# 02 — MAML Toy Verification

**Gate before touching real Wav2Vec2:**
- `higher` wraps a toy MLP and produces second-order gradients
- Full MAML gradient ≠ FOMAML gradient on identical input (confirms Hessian term)
- Query loss decreases as k increases on synthetic tasks
- `higher` wraps actual `Wav2Vec2ForCTC` on CPU without error
- Memory usage per step logged (confirm 12GB not exceeded for MLP)

**Architecture:** Toy MLP (input=768, hidden=256, output=32) mimicking Wav2Vec2 encoder + lm_head structure.
Random tensors used as synthetic 'audio features'.

In [1]:
import torch
import torch.nn as nn
import higher
import copy
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.1f} GB / {total/1e9:.1f} GB')

Device: cuda
VRAM free: 11.4 GB / 12.4 GB


## 1. Toy MLP — mimics Wav2Vec2 encoder + lm_head

In [2]:
class ToyModel(nn.Module):
    """Mimics Wav2Vec2: encoder (large) + lm_head (small, personalization)."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.lm_head = nn.Linear(256, 32)

    def forward(self, x):
        return self.lm_head(self.encoder(x))

    def get_inner_loop_params(self):
        return list(self.lm_head.parameters())

model = ToyModel().to(device)
total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.lm_head.parameters())
enc = sum(p.numel() for name, p in model.named_parameters() if 'lm_head' not in name)
print(f'Total params: {total:,}')
print(f'Encoder:      {enc:,}  (outer loop)')
print(f'lm_head:      {head:,}  (inner loop — ANIL)')

Total params: 533,280
Encoder:      525,056  (outer loop)
lm_head:      8,224  (inner loop — ANIL)


## 2. Synthetic task data

In [3]:
def make_task(batch=4, dim=768, seed=None):
    """Random support/query tensors mimicking audio feature batches."""
    if seed is not None:
        torch.manual_seed(seed)
    support_x = torch.randn(batch, dim, device=device)
    support_y = torch.randint(0, 32, (batch,), device=device)
    query_x = torch.randn(batch, dim, device=device)
    query_y = torch.randint(0, 32, (batch,), device=device)
    return support_x, support_y, query_x, query_y

loss_fn = nn.CrossEntropyLoss()
print('Synthetic task data ready')

Synthetic task data ready


## 3. Full MAML via `higher` (second-order)

In [4]:
def full_maml_grad(model, support_x, support_y, query_x, query_y, k=3, inner_lr=1e-2):
    inner_opt = torch.optim.SGD(model.get_inner_loop_params(), lr=inner_lr)
    
    with higher.innerloop_ctx(
        model, inner_opt,
        copy_initial_weights=False,
        track_higher_grads=True  # SECOND-ORDER: Hessian term included
    ) as (fmodel, diffopt):
        for _ in range(k):
            out = fmodel(support_x)
            loss = loss_fn(out, support_y)
            diffopt.step(loss)
        query_out = fmodel(query_x)
        query_loss = loss_fn(query_out, query_y)
    
    grads = torch.autograd.grad(
        query_loss, model.parameters(),
        allow_unused=True, retain_graph=False
    )
    return list(grads), query_loss.item()

sup_x, sup_y, qry_x, qry_y = make_task(seed=42)
full_grads, full_loss = full_maml_grad(model, sup_x, sup_y, qry_x, qry_y)
print(f'Full MAML query loss: {full_loss:.4f}')
print(f'Non-None grads: {sum(1 for g in full_grads if g is not None)}/{len(full_grads)}')

Full MAML query loss: 3.4793
Non-None grads: 6/6


## 4. FOMAML (first-order, native PyTorch)

In [5]:
def fomaml_grad(model, support_x, support_y, query_x, query_y, k=3, inner_lr=1e-2):
    init_state = copy.deepcopy(model.state_dict())
    inner_opt = torch.optim.SGD(model.get_inner_loop_params(), lr=inner_lr)
    
    for _ in range(k):
        inner_opt.zero_grad()
        out = model(support_x)
        loss = loss_fn(out, support_y)
        loss.backward()
        inner_opt.step()
    
    model.zero_grad()
    query_out = model(query_x)
    query_loss = loss_fn(query_out, query_y)
    
    grads = torch.autograd.grad(
        query_loss, model.parameters(),
        allow_unused=True, retain_graph=False,
        create_graph=False  # FIRST-ORDER: no Hessian
    )
    q_loss_val = query_loss.item()
    model.load_state_dict(init_state)
    return list(grads), q_loss_val

fo_grads, fo_loss = fomaml_grad(model, sup_x, sup_y, qry_x, qry_y)
print(f'FOMAML query loss: {fo_loss:.4f}')

FOMAML query loss: 3.4793


## 5. Critical gate: Full MAML grad ≠ FOMAML grad

In [ ]:
valid_pairs = [
    (f, g) for f, g in zip(full_grads, fo_grads)
    if f is not None and g is not None
]

max_diff = max(float((f - g).abs().max()) for f, g in valid_pairs)
mean_diff = np.mean([float((f - g).abs().mean()) for f, g in valid_pairs])
any_different = any(not torch.allclose(f, g, atol=1e-6) for f, g in valid_pairs)

print(f'Max gradient difference:  {max_diff:.6f}')
print(f'Mean gradient difference: {mean_diff:.6f}')
print(f'Gradients differ: {any_different}')

if any_different:
    print('✓ GATE PASSED: Full MAML grad ≠ FOMAML grad — second-order computation verified')
else:
    print('✗ GATE FAILED: Gradients identical — higher not computing second-order term')

## 6. Adaptation curves: query loss at k=0,1,3,5

In [ ]:
k_values = [0, 1, 3, 5]
full_losses = []
fo_losses = []

sup_x, sup_y, qry_x, qry_y = make_task(seed=99)

for k in k_values:
    _, fl = full_maml_grad(model, sup_x, sup_y, qry_x, qry_y, k=k)
    _, fol = fomaml_grad(model, sup_x, sup_y, qry_x, qry_y, k=k)
    full_losses.append(fl)
    fo_losses.append(fol)
    print(f'k={k}: Full MAML={fl:.4f}  FOMAML={fol:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(k_values, full_losses, 'b-o', label='Full MAML (2nd order)')
plt.plot(k_values, fo_losses, 'r--s', label='FOMAML (1st order)')
plt.xlabel('Inner loop steps k')
plt.ylabel('Query loss')
plt.title('Adaptation curves: query loss vs k (toy MLP)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('notebooks/02_adaptation_curves.png', dpi=100)
plt.show()
print('Adaptation curves saved: notebooks/02_adaptation_curves.png')

## 7. Memory profiling (confirm 12GB VRAM not exceeded)

In [ ]:
if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    _, _ = full_maml_grad(model, sup_x, sup_y, qry_x, qry_y, k=3)
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f'Peak VRAM (full MAML, k=3, toy MLP): {peak_mb:.1f} MB')
    print(f'12GB budget: {peak_mb/1e3:.3f} GB / 12 GB used')
    assert peak_mb < 12000, f'Peak VRAM {peak_mb:.0f}MB exceeds 12GB budget'
    print('✓ Memory budget OK')
else:
    print('CUDA not available — skipping VRAM check')

## 8. Critical gate: `higher` wraps actual Wav2Vec2ForCTC on CPU

In [ ]:
# This runs on CPU in seconds — must pass before paying for A100 time
from transformers import Wav2Vec2ForCTC

print('Loading facebook/wav2vec2-base-960h on CPU...')
wav2vec2 = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h')
inner_opt_w2v = torch.optim.SGD(wav2vec2.lm_head.parameters(), lr=1e-4)

# 1 second of fake audio (no GPU needed for this check)
dummy = torch.randn(1, 16000)

try:
    with higher.innerloop_ctx(wav2vec2, inner_opt_w2v, track_higher_grads=True) as (fm, dopt):
        out = fm(input_values=dummy)
        dopt.step(out.logits.mean())  # dummy loss — no labels needed for this check
    print('✓ GATE PASSED: higher wraps Wav2Vec2ForCTC cleanly')
    print('  Safe to proceed to A100 for full MAML experiments')
except Exception as e:
    print(f'✗ GATE FAILED: {e}')
    print('  Debug before booking A100 time')

## 9. Summary

In [ ]:
print('=== Phase 0 Verification Summary ===')
print(f'1. higher wraps toy MLP:        ✓')
print(f'2. Full MAML grad ≠ FOMAML grad: {"✓" if any_different else "✗"}')
print(f'3. Query loss decreases with k: {"✓" if fo_losses[-1] < fo_losses[0] else "check results"}')
print(f'4. higher wraps Wav2Vec2:       (see cell 8 above)')
print()
print('If all gates pass, proceed to:')
print('  python data/pii_masking.py')
print('  python data/features.py')
print('  python data/partition.py')